In [ ]:
!pip install torch transformers datasets

# GPT

In [ ]:
# =============================================================
# SUPERPOSITION TRANSFORMER (ST-v1) — FINAL CLEAN VERSION
# =============================================================

import torch
import torch.nn as nn
import time
import re

from transformers import AutoModelForCausalLM, AutoTokenizer
from datasets import load_dataset
from torch.utils.data import DataLoader

# =============================================================
# 1. DEVICE
# =============================================================
device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# =============================================================
# 2. LOAD MODEL
# =============================================================
model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# FIX: ensure pad token exists
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    dtype=torch.float16 if device == "cuda" else torch.float32,
    device_map="auto"
)

base_model.eval()

# Freeze backbone
for p in base_model.parameters():
    p.requires_grad = False

# =============================================================
# 3. SUPERPOSITION LAYER
# =============================================================
class SuperpositionLayer(nn.Module):
    def __init__(self, hidden_dim, K=4):
        super().__init__()

        self.K = K

        # Entanglement mixing matrix
        self.mixing = nn.Parameter(torch.randn(K, K) * 0.02)

        self.proj = nn.Linear(hidden_dim, hidden_dim)
        self.gate = nn.Linear(hidden_dim, 1)
        self.norm = nn.LayerNorm(hidden_dim)

    def forward(self, h):
        B, T, D = h.shape

        # 1. SUPERPOSITION (K hypotheses)
        Psi = h.unsqueeze(2).repeat(1, 1, self.K, 1)

        # Add noise to diversify hypotheses
        Psi = Psi + 0.01 * torch.randn_like(Psi)

        # 2. ENTANGLEMENT (interaction)
        Psi = torch.einsum("hk, btkd -> bthd", self.mixing, Psi)

        # Residual
        Psi = Psi + h.unsqueeze(2)

        # 3. INTERFERENCE
        Psi = torch.tanh(Psi)
        Psi = Psi / (Psi.norm(dim=2, keepdim=True) + 1e-6)

        # 4. COLLAPSE
        weights = torch.softmax(self.gate(Psi).squeeze(-1), dim=2)
        h_out = torch.sum(Psi * weights.unsqueeze(-1), dim=2)

        h_out = self.proj(h_out)
        h_out = self.norm(h_out)

        return h_out


# =============================================================
# 4. MODEL WRAPPER
# =============================================================
class SuperpositionModel(nn.Module):
    def __init__(self, base_model, hidden_dim, K=4):
        super().__init__()

        self.base = base_model
        self.super_layer = SuperpositionLayer(hidden_dim, K)

    def forward(self, input_ids, attention_mask=None, labels=None):

        device = input_ids.device

        outputs = self.base.model(
            input_ids=input_ids.to(device),
            attention_mask=attention_mask.to(device) if attention_mask is not None else None
        )

        h = outputs.last_hidden_state

        # Apply ST reasoning
        h = self.super_layer(h)

        logits = self.base.lm_head(h)

        loss = None
        if labels is not None:
            loss_fct = nn.CrossEntropyLoss(ignore_index=-100)

            shift_logits = logits[:, :-1, :].contiguous()
            shift_labels = labels[:, 1:].contiguous()

            loss = loss_fct(
                shift_logits.view(-1, shift_logits.size(-1)),
                shift_labels.view(-1)
            )

        return {"loss": loss, "logits": logits}


# =============================================================
# 5. INIT MODEL
# =============================================================
hidden_dim = base_model.config.hidden_size
model = SuperpositionModel(base_model, hidden_dim, K=4)

# =============================================================
# 6. DATASET
# =============================================================
dataset = load_dataset("gsm8k", "main", split="train[:200]")

def preprocess(example):
    prompt = example["question"] + "\nSolve step by step.\nFinal Answer:"
    answer = example["answer"]

    full_text = prompt + " " + answer

    enc = tokenizer(
        full_text,
        truncation=True,
        padding="max_length",
        max_length=192
    )

    input_ids = enc["input_ids"]
    labels = input_ids.copy()

    # Correct prompt length
    prompt_ids = tokenizer(
        prompt,
        truncation=True,
        max_length=192
    )["input_ids"]

    prompt_len = len(prompt_ids)

    # Mask prompt tokens
    labels[:prompt_len] = [-100] * prompt_len

    enc["labels"] = labels
    return enc


dataset = dataset.map(preprocess)
dataset.set_format(type="torch", columns=["input_ids", "attention_mask", "labels"])

loader = DataLoader(dataset, batch_size=4, shuffle=True)

# =============================================================
# 7. TRAINING
# =============================================================
optimizer = torch.optim.AdamW(model.super_layer.parameters(), lr=1e-4)

model.train()

for epoch in range(5):
    total_loss = 0

    for batch in loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        out = model(input_ids, attention_mask, labels)
        loss = out["loss"]

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader):.4f}")


# =============================================================
# 8. HELPERS
# =============================================================
def extract_number(text):
    nums = re.findall(r"\d+", text)
    return nums[-1] if nums else None


# =============================================================
# 9. GENERATION
# =============================================================
def generate_classical(prompt):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    outputs = base_model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False
    )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)


def generate_st(prompt):
    model.eval()

    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    input_ids = inputs["input_ids"]

    for _ in range(50):
        out = model(input_ids)

        logits = out["logits"][:, -1, :]
        next_token = torch.argmax(logits, dim=-1, keepdim=True)

        input_ids = torch.cat([input_ids, next_token], dim=1)

    return tokenizer.decode(input_ids[0], skip_special_tokens=True)


# =============================================================
# 10. EVALUATION
# =============================================================
test_dataset = load_dataset("gsm8k", "main", split="test[:20]")

def evaluate():
    cl_correct = 0
    st_correct = 0

    cl_times = []
    st_times = []

    for i, sample in enumerate(test_dataset):

        question = sample["question"]
        gt = extract_number(sample["answer"])

        prompt = question + "\nSolve step by step.\nFinal Answer:"

        # Classical
        t1 = time.time()
        cl_out = generate_classical(prompt)
        t2 = time.time()

        # ST
        t3 = time.time()
        st_out = generate_st(prompt)
        t4 = time.time()

        cl_ans = extract_number(cl_out)
        st_ans = extract_number(st_out)

        if cl_ans == gt:
            cl_correct += 1
        if st_ans == gt:
            st_correct += 1

        cl_times.append(t2 - t1)
        st_times.append(t4 - t3)

        print(f"\nSample {i+1}")
        print("GT:", gt)
        print("Classical:", cl_ans)
        print("ST:", st_ans)

    print("\n===== FINAL RESULTS =====")
    print("Classical Accuracy:", cl_correct / len(test_dataset))
    print("ST Accuracy:", st_correct / len(test_dataset))
    print("Classical Time:", sum(cl_times)/len(cl_times))
    print("ST Time:", sum(st_times)/len(st_times))


# Run
evaluate()

Using device: cpu


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Epoch 1, Loss: 3.8137
Epoch 2, Loss: 2.1390
Epoch 3, Loss: 1.6155
Epoch 4, Loss: 1.2742
Epoch 5, Loss: 1.0283

Sample 1
GT: 18
Classical: 120
ST: 15

Sample 2
GT: 3
Classical: 6
ST: 2

Sample 3
GT: 70000
Classical: 000
ST: 15

Sample 4
GT: 540
Classical: 4
ST: 120

Sample 5
GT: 20
Classical: 3
ST: 25

Sample 6
GT: 64
Classical: 1
ST: 20

Sample 7
GT: 260
Classical: 2
ST: 128

Sample 8
GT: 160
Classical: 20
ST: 20
